##STEP 1 — Mount Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path

RAW_DIR = Path(
    "/content/drive/MyDrive/Image_captioning/Dataset/Raw"
)

print("RAW folder contents:\n")

for item in RAW_DIR.iterdir():
    print(repr(item.name))

RAW folder contents:

'Flickr8K'
'Flickr30k'


STEP 2 — Imports + Paths

In [ ]:
# ============================================================
# NOTEBOOK 2 — SETUP
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import re


PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Image_captioning"
)

DATASET_DIR = PROJECT_ROOT / "Dataset"

RAW_DIR = DATASET_DIR / "Raw"
PROCESSED_DIR = DATASET_DIR / "Processed"

FLICKR8K_RAW_DIR = RAW_DIR / "Flickr8K"
FLICKR30K_RAW_DIR = RAW_DIR / "Flickr30k"


print("=" * 70)
print("NOTEBOOK 2 — DATASET MERGING")
print("=" * 70)

print("\nFlickr8K:")
print(FLICKR8K_RAW_DIR)

print("\nFlickr30K:")
print(FLICKR30K_RAW_DIR)


assert FLICKR8K_RAW_DIR.exists()
assert FLICKR30K_RAW_DIR.exists()


print("\n✓ Flickr8K path verified")
print("✓ Flickr30K path verified")
print("\n✓ Notebook 2 setup ready")

NOTEBOOK 2 — DATASET MERGING

Flickr8K:
/content/drive/MyDrive/Image_captioning/Dataset/Raw/Flickr8K

Flickr30K:
/content/drive/MyDrive/Image_captioning/Dataset/Raw/Flickr30k

✓ Flickr8K path verified
✓ Flickr30K path verified

✓ Notebook 2 setup ready


STEP 3 — Find Caption / Annotation Files

In [ ]:
# ============================================================
# FIND TEXT / ANNOTATION FILES
# ============================================================

TEXT_EXTENSIONS = {
    ".txt",
    ".csv",
    ".json",
    ".xml"
}


def find_text_files(directory):

    return sorted(
        [
            file
            for file in directory.rglob("*")
            if (
                file.is_file()
                and file.suffix.lower()
                in TEXT_EXTENSIONS
            )
        ]
    )


flickr8k_text_files = find_text_files(
    FLICKR8K_RAW_DIR
)

flickr30k_text_files = find_text_files(
    FLICKR30K_RAW_DIR
)


print("=" * 70)
print("FLICKR8K TEXT / ANNOTATION FILES")
print("=" * 70)

for file in flickr8k_text_files:
    print(
        file.relative_to(
            FLICKR8K_RAW_DIR
        )
    )


print("\n" + "=" * 70)
print("FLICKR30K TEXT / ANNOTATION FILES")
print("=" * 70)

for file in flickr30k_text_files:
    print(
        file.relative_to(
            FLICKR30K_RAW_DIR
        )
    )


print("\n" + "=" * 70)
print("FILE SEARCH COMPLETE")
print("=" * 70)

print(
    f"\nFlickr8K text files : "
    f"{len(flickr8k_text_files)}"
)

print(
    f"Flickr30K text files: "
    f"{len(flickr30k_text_files)}"
)

FLICKR8K TEXT / ANNOTATION FILES
captions.txt

FLICKR30K TEXT / ANNOTATION FILES
flickr30k/results.csv

FILE SEARCH COMPLETE

Flickr8K text files : 1
Flickr30K text files: 1


STEP 4: Inspect Caption Files

In [ ]:
# ============================================================
# INSPECT FLICKR30K RESULTS.CSV RAW FORMAT
# ============================================================

print("=" * 70)
print("FLICKR30K RAW FILE INSPECTION")
print("=" * 70)

print("\nFile:")
print(FLICKR30K_CAPTIONS_FILE)

print("\nFirst 15 raw lines:\n")

with open(
    FLICKR30K_CAPTIONS_FILE,
    "r",
    encoding="utf-8",
    errors="replace"
) as file:

    for i in range(15):

        line = file.readline()

        if not line:
            break

        print(
            f"Line {i + 1}: {repr(line.rstrip())}"
        )


print("\n" + "=" * 70)
print("DELIMITER CHECK")
print("=" * 70)

with open(
    FLICKR30K_CAPTIONS_FILE,
    "r",
    encoding="utf-8",
    errors="replace"
) as file:

    lines = []

    for _ in range(10):

        line = file.readline()

        if line:
            lines.append(line.rstrip())


for delimiter in [
    ",",
    ";",
    "\t",
    "|"
]:

    counts = [
        line.count(delimiter)
        for line in lines
    ]

    print(
        repr(delimiter),
        "counts:",
        counts
    )


print("\n" + "=" * 70)
print("RAW INSPECTION COMPLETE")
print("=" * 70)

FLICKR30K RAW FILE INSPECTION

File:
/content/drive/MyDrive/Image_captioning/Dataset/Raw/Flickr30k/flickr30k/results.csv

First 15 raw lines:

Line 1: 'image_name| comment_number| comment'
Line 2: '1000092795.jpg| 0| Two young guys with shaggy hair look at their hands while hanging out in the yard .'
Line 3: '1000092795.jpg| 1| Two young , White males are outside near many bushes .'
Line 4: '1000092795.jpg| 2| Two men in green shirts are standing in a yard .'
Line 5: '1000092795.jpg| 3| A man in a blue shirt standing in a garden .'
Line 6: '1000092795.jpg| 4| Two friends enjoy time spent together .'
Line 7: '10002456.jpg| 0| Several men in hard hats are operating a giant pulley system .'
Line 8: '10002456.jpg| 1| Workers look down from up above on a piece of equipment .'
Line 9: '10002456.jpg| 2| Two men working on a machine wearing hard hats .'
Line 10: '10002456.jpg| 3| Four men on top of a tall structure .'
Line 11: '10002456.jpg| 4| Three men on a large rig .'
Line 12: '1000268201.

STEP 5: Load and Verify Both Datasets

In [ ]:
# ============================================================
# STEP 5 — LOAD AND VERIFY BOTH DATASETS
# ============================================================

print("=" * 70)
print("LOADING FLICKR8K")
print("=" * 70)

# Flickr8K is a normal comma-separated file
flickr8k_df = pd.read_csv(
    FLICKR8K_CAPTIONS_FILE
)

print("\nColumns:")
print(list(flickr8k_df.columns))

print("\nShape:")
print(flickr8k_df.shape)

print("\nFirst 5 rows:")
print(
    flickr8k_df.head().to_string(
        index=False
    )
)


# ============================================================
# FLICKR30K
# ============================================================

print("\n" + "=" * 70)
print("LOADING FLICKR30K")
print("=" * 70)

# IMPORTANT:
# results.csv is pipe-separated, not comma-separated.
flickr30k_df = pd.read_csv(
    FLICKR30K_CAPTIONS_FILE,
    sep="|",
    skipinitialspace=True
)

# Remove accidental spaces from column names
flickr30k_df.columns = [
    column.strip()
    for column in flickr30k_df.columns
]

# Remove leading/trailing spaces from text columns
for column in flickr30k_df.columns:

    if (
        flickr30k_df[column].dtype
        == "object"
    ):
        flickr30k_df[column] = (
            flickr30k_df[column]
            .astype(str)
            .str.strip()
        )


print("\nColumns:")
print(list(flickr30k_df.columns))

print("\nShape:")
print(flickr30k_df.shape)

print("\nFirst 5 rows:")
print(
    flickr30k_df.head().to_string(
        index=False
    )
)


# ============================================================
# BASIC VERIFICATION
# ============================================================

print("\n" + "=" * 70)
print("DATASET VERIFICATION")
print("=" * 70)

print(
    "\nFlickr8K caption rows:",
    len(flickr8k_df)
)

print(
    "Flickr8K unique images:",
    flickr8k_df["image"].nunique()
)

print(
    "\nFlickr30K caption rows:",
    len(flickr30k_df)
)

print(
    "Flickr30K unique images:",
    flickr30k_df["image_name"].nunique()
)


print("\nMissing values:")

print(
    "\nFlickr8K:"
)
print(
    flickr8k_df.isna().sum()
)

print(
    "\nFlickr30K:"
)
print(
    flickr30k_df.isna().sum()
)


print("\n" + "=" * 70)
print("✓ BOTH DATASETS LOADED SUCCESSFULLY")
print("=" * 70)

LOADING FLICKR8K

Columns:
['image', 'caption']

Shape:
(40455, 2)

First 5 rows:
                    image                                                                  caption
1000268201_693b08cb0e.jpg A child in a pink dress is climbing up a set of stairs in an entry way .
1000268201_693b08cb0e.jpg                                    A girl going into a wooden building .
1000268201_693b08cb0e.jpg                         A little girl climbing into a wooden playhouse .
1000268201_693b08cb0e.jpg                     A little girl climbing the stairs to her playhouse .
1000268201_693b08cb0e.jpg                A little girl in a pink dress going into a wooden cabin .

LOADING FLICKR30K

Columns:
['image_name', 'comment_number', 'comment']

Shape:
(158915, 3)

First 5 rows:
    image_name comment_number                                                                             comment
1000092795.jpg              0 Two young guys with shaggy hair look at their hands while hanging out in

STEP 6: Standardize Both Datasets

In [ ]:
# ============================================================
# STEP 6 — STANDARDIZE DATASET FORMAT
# ============================================================

print("=" * 70)
print("STANDARDIZING DATASET FORMATS")
print("=" * 70)


# ------------------------------------------------------------
# FLICKR8K
# ------------------------------------------------------------

flickr8k_standard = pd.DataFrame({
    "dataset": "Flickr8K",
    "image_name": flickr8k_df["image"].astype(str).str.strip(),
    "caption": flickr8k_df["caption"].astype(str).str.strip()
})


# ------------------------------------------------------------
# FLICKR30K
# ------------------------------------------------------------

flickr30k_standard = pd.DataFrame({
    "dataset": "Flickr30K",
    "image_name": flickr30k_df["image_name"].astype(str).str.strip(),
    "caption": flickr30k_df["comment"].astype(str).str.strip()
})


# ------------------------------------------------------------
# VERIFY
# ------------------------------------------------------------

print("\nFlickr8K standardized:")
print(flickr8k_standard.shape)

print("\nFlickr30K standardized:")
print(flickr30k_standard.shape)


print("\nFlickr8K sample:")
print(
    flickr8k_standard.head().to_string(
        index=False
    )
)


print("\nFlickr30K sample:")
print(
    flickr30k_standard.head().to_string(
        index=False
    )
)


# ------------------------------------------------------------
# COLUMN VERIFICATION
# ------------------------------------------------------------

EXPECTED_COLUMNS = [
    "dataset",
    "image_name",
    "caption"
]


assert list(
    flickr8k_standard.columns
) == EXPECTED_COLUMNS

assert list(
    flickr30k_standard.columns
) == EXPECTED_COLUMNS


print("\n✓ Flickr8K standardized")
print("✓ Flickr30K standardized")
print("✓ Column formats identical")

STANDARDIZING DATASET FORMATS

Flickr8K standardized:
(40455, 3)

Flickr30K standardized:
(158915, 3)

Flickr8K sample:
 dataset                image_name                                                                  caption
Flickr8K 1000268201_693b08cb0e.jpg A child in a pink dress is climbing up a set of stairs in an entry way .
Flickr8K 1000268201_693b08cb0e.jpg                                    A girl going into a wooden building .
Flickr8K 1000268201_693b08cb0e.jpg                         A little girl climbing into a wooden playhouse .
Flickr8K 1000268201_693b08cb0e.jpg                     A little girl climbing the stairs to her playhouse .
Flickr8K 1000268201_693b08cb0e.jpg                A little girl in a pink dress going into a wooden cabin .

Flickr30K sample:
  dataset     image_name                                                                             caption
Flickr30K 1000092795.jpg Two young guys with shaggy hair look at their hands while hanging out in the ya

STEP 7: Create Safe Unique Image IDs

In [ ]:
# ============================================================
# STEP 7 — CREATE UNIQUE IMAGE IDENTIFIERS
# ============================================================

print("=" * 70)
print("CREATING UNIQUE IMAGE IDENTIFIERS")
print("=" * 70)


def add_image_id(dataframe):

    dataframe = dataframe.copy()

    dataframe["image_id"] = (
        dataframe["dataset"]
        + "/"
        + dataframe["image_name"]
    )

    return dataframe


flickr8k_standard = add_image_id(
    flickr8k_standard
)

flickr30k_standard = add_image_id(
    flickr30k_standard
)


# Put image_id first for easier inspection
COLUMN_ORDER = [
    "image_id",
    "dataset",
    "image_name",
    "caption"
]


flickr8k_standard = (
    flickr8k_standard[
        COLUMN_ORDER
    ]
)

flickr30k_standard = (
    flickr30k_standard[
        COLUMN_ORDER
    ]
)


print("\nFlickr8K sample:\n")

print(
    flickr8k_standard.head().to_string(
        index=False
    )
)


print("\nFlickr30K sample:\n")

print(
    flickr30k_standard.head().to_string(
        index=False
    )
)


assert (
    flickr8k_standard["image_id"]
    .str.startswith("Flickr8K/")
    .all()
)

assert (
    flickr30k_standard["image_id"]
    .str.startswith("Flickr30K/")
    .all()
)


print("\n✓ Unique image IDs created safely")

CREATING UNIQUE IMAGE IDENTIFIERS

Flickr8K sample:

                          image_id  dataset                image_name                                                                  caption
Flickr8K/1000268201_693b08cb0e.jpg Flickr8K 1000268201_693b08cb0e.jpg A child in a pink dress is climbing up a set of stairs in an entry way .
Flickr8K/1000268201_693b08cb0e.jpg Flickr8K 1000268201_693b08cb0e.jpg                                    A girl going into a wooden building .
Flickr8K/1000268201_693b08cb0e.jpg Flickr8K 1000268201_693b08cb0e.jpg                         A little girl climbing into a wooden playhouse .
Flickr8K/1000268201_693b08cb0e.jpg Flickr8K 1000268201_693b08cb0e.jpg                     A little girl climbing the stairs to her playhouse .
Flickr8K/1000268201_693b08cb0e.jpg Flickr8K 1000268201_693b08cb0e.jpg                A little girl in a pink dress going into a wooden cabin .

Flickr30K sample:

                image_id   dataset     image_name                    

STEP 8: Merge the Caption Records

In [ ]:
# ============================================================
# STEP 8 — MERGE FLICKR8K + FLICKR30K
# ============================================================

print("=" * 70)
print("MERGING DATASETS")
print("=" * 70)


merged_df = pd.concat(
    [
        flickr8k_standard,
        flickr30k_standard
    ],
    axis=0,
    ignore_index=True
)


print("\nMerged shape:")
print(merged_df.shape)


print("\nRecords by dataset:")

print(
    merged_df["dataset"]
    .value_counts()
)


print("\nUnique images by dataset:")

print(
    merged_df.groupby(
        "dataset"
    )["image_id"]
    .nunique()
)


print(
    "\nTotal caption records:",
    len(merged_df)
)

print(
    "Total unique image IDs:",
    merged_df["image_id"].nunique()
)


print("\nMissing values:")

print(
    merged_df.isna().sum()
)


assert len(
    merged_df
) == (
    len(flickr8k_standard)
    + len(flickr30k_standard)
)


assert (
    merged_df["image_id"].notna().all()
)

assert (
    merged_df["caption"].notna().all()
)


print("\n✓ Datasets merged successfully")

MERGING DATASETS

Merged shape:
(199370, 4)

Records by dataset:
dataset
Flickr30K    158915
Flickr8K      40455
Name: count, dtype: int64

Unique images by dataset:
dataset
Flickr30K    31783
Flickr8K      8091
Name: image_id, dtype: int64

Total caption records: 199370
Total unique image IDs: 39874

Missing values:
image_id      0
dataset       0
image_name    0
caption       0
dtype: int64

✓ Datasets merged successfully


STEP 9: Save the Unified Manifest

In [ ]:
# ============================================================
# STEP 9 — SAVE UNIFIED DATASET MANIFEST
# ============================================================

MERGED_DIR = (
    PROCESSED_DIR
    / "Merged"
)

MERGED_DIR.mkdir(
    parents=True,
    exist_ok=True
)


MERGED_MANIFEST_PATH = (
    MERGED_DIR
    / "merged_captions_raw.csv"
)


print("=" * 70)
print("SAVING UNIFIED MANIFEST")
print("=" * 70)

print("\nOutput:")
print(MERGED_MANIFEST_PATH)


merged_df.to_csv(
    MERGED_MANIFEST_PATH,
    index=False
)


print("\nSaved successfully.")

print(
    f"\nFile size: "
    f"{MERGED_MANIFEST_PATH.stat().st_size / (1024 * 1024):.2f} MB"
)


print(
    "\n✓ Unified raw caption manifest saved"
)

SAVING UNIFIED MANIFEST

Output:
/content/drive/MyDrive/Image_captioning/Dataset/Processed/Merged/merged_captions_raw.csv

Saved successfully.

File size: 22.27 MB

✓ Unified raw caption manifest saved


STEP 10: Final Verification

In [ ]:
# ============================================================
# CHECK MISSING CAPTIONS
# ============================================================

print("=" * 70)
print("MISSING CAPTION INVESTIGATION")
print("=" * 70)

missing_caption_rows = verification_df[
    verification_df["caption"].isna()
]

print(
    "\nNumber of missing captions:",
    len(missing_caption_rows)
)

if len(missing_caption_rows) > 0:

    print("\nMissing caption rows:\n")

    print(
        missing_caption_rows[
            [
                "image_id",
                "dataset",
                "image_name",
                "caption"
            ]
        ].head(20).to_string(index=False)
    )

    print("\nMissing captions by dataset:\n")

    print(
        missing_caption_rows[
            "dataset"
        ].value_counts()
    )

else:
    print("\n✓ No missing captions found")

MISSING CAPTION INVESTIGATION

Number of missing captions: 1

Missing caption rows:

                image_id   dataset     image_name caption
Flickr30K/2199200615.jpg Flickr30K 2199200615.jpg     NaN

Missing captions by dataset:

dataset
Flickr30K    1
Name: count, dtype: int64


In [ ]:
# ============================================================
# FIX SINGLE MISSING CAPTION
# ============================================================

print("=" * 70)
print("FIXING MISSING CAPTION")
print("=" * 70)

print("\nBefore cleaning:")
print("Total rows:", len(verification_df))
print(
    "Missing captions:",
    verification_df["caption"].isna().sum()
)


# Remove rows with missing or empty captions
verification_df = verification_df.dropna(
    subset=["caption"]
).copy()

verification_df["caption"] = (
    verification_df["caption"]
    .astype(str)
    .str.strip()
)

verification_df = verification_df[
    verification_df["caption"] != ""
].copy()


print("\nAfter cleaning:")
print("Total rows:", len(verification_df))
print(
    "Missing captions:",
    verification_df["caption"].isna().sum()
)


# Expected: 199,369 rows
assert (
    verification_df["caption"].notna().all()
)

assert (
    verification_df["caption"].str.len() > 0
).all()


print("\n✓ Missing/empty captions removed")
print("✓ Caption data is clean")

FIXING MISSING CAPTION

Before cleaning:
Total rows: 199370
Missing captions: 1

After cleaning:
Total rows: 199369
Missing captions: 0

✓ Missing/empty captions removed
✓ Caption data is clean


In [ ]:
# ============================================================
# SAVE CLEANED MERGED MANIFEST
# ============================================================

verification_df.to_csv(
    MERGED_MANIFEST_PATH,
    index=False
)

print("=" * 70)
print("CLEAN MANIFEST SAVED")
print("=" * 70)

print("\nPath:")
print(MERGED_MANIFEST_PATH)

print("\nFinal caption records:")
print(len(verification_df))

print("\nFinal unique images:")
print(
    verification_df["image_id"].nunique()
)

print("\nMissing captions:")
print(
    verification_df["caption"].isna().sum()
)

print("\n✓ Clean merged dataset saved successfully")

CLEAN MANIFEST SAVED

Path:
/content/drive/MyDrive/Image_captioning/Dataset/Processed/Merged/merged_captions_raw.csv

Final caption records:
199369

Final unique images:
39874

Missing captions:
0

✓ Clean merged dataset saved successfully
